# **Chapter 15: Using Databases and SQL**

## **15.1 What is a database?**

A database is a file that is organized for storing data. Most databases are organized like a dictionary in the sense that they map from keys to values. The biggest difference is that the database is on disk (or other permanent storage), so it persists after the program ends.

Database software maintains its performance by building indexes as data is added to the database to allow the computer to jump quickly to a particular entry.

There are many different database systems which are used for a wide variety of pur poses including: Oracle, MySQL, Microsoft SQL Server, PostgreSQL, and SQLite. We focus on SQLite

## **15.2 Database concepts**

When you first look at a database it looks like a spreadsheet with multiple sheets. The primary data structures in a database are: tables, rows, and columns.

In technical descriptions of relational databases the concepts of table, row, and column are more formally referred to as relation, tuple, and attribute, respectively. We will use the less formal terms in this chapter.

## **15.3 Database Browser for SQLite**

While this chapter will focus on using Python to work with data in SQLite database files, many operations can be done more conveniently using software called the Database Browser for SQLite which is freely available from:

http://sqlitebrowser.org/

Using the browser you can easily create tables, insert data, edit data, or run simple SQL queries on the data in the database. In a sense, the database browser is similar to a text editor when working with text files.

## **15.4 Creating a database table**

Databases require more defined structure than Python lists or dictionaries.

When we create a database table we must tell the database in advance the names
of each of the columns in the table and the type of data which we are planning to
store in each column. When the database software knows the type of data in each
column, it can choose the most efficient way to store and look up the data based
on the type of data.

In [1]:
import sqlite3

conn = sqlite3.connect('music.sqlite')
cur = conn.cursor()

cur.execute('DROP TABLE IF EXISTS Track')
cur.execute('CREATE TABLE Track (title TEXT, plays INTEGER)')
conn.close()

The `connect` operation makes a “connection” to the database stored in the file
`music.sqlite` in the current directory. If the file does not exist, it will be created.
The reason this is called a “connection” is that sometimes the database is stored
on a separate “database server” from the server on which we are running our
application.

A cursor is like a file handle that we can use to perform operations on the data
stored in the database. Calling `cursor()` is very similar conceptually to calling
`open()` when dealing with text files.

Once we have the cursor, we can begin to execute commands on the contents of
the database using the `execute()` method.

Database commands are expressed in a special language that has been standardized
across many different database vendors to allow us to learn a single database
language. The database language is called Structured Query Language or SQL for
short.

The first SQL command removes the `Track` table from the database if it exists.
This pattern is simply to allow us to run the same program to create the Track
table over and over again without causing an error. The second command creates a table named Track with a text column named `title` and an integer column named `plays`.

In [2]:
import sqlite3

conn = sqlite3.connect('music.sqlite')
cur = conn.cursor()

cur.execute('INSERT INTO Track (title, plays) VALUES (?, ?)',
            ('Thunderstruck', 20))
cur.execute('INSERT INTO Track (title, plays) VALUES (?, ?)',
            ('My Way', 15))
conn.commit()

print('Track:')
cur.execute('SELECT title, plays FROM Track')
for row in cur:
    print(row)

cur.execute('DELETE FROM Track WHERE plays < 100')
conn.commit()
cur.close()

Track:
('Thunderstruck', 20)
('My Way', 15)


The SQL `INSERT` command indicates which table we are using and then defines a new row by listing the fields we want to include (`title`, `plays`) followed by the `VALUES` we want placed in the new row. We specify the values as question marks `(?, ?)` to indicate that the actual values are passed in as a tuple ( `'My Way'`, `15`) as the second parameter to the `execute()` call.

First we `INSERT` two rows into our table and use `commit()` to force the data to be
written to the database file. Then we use the SELECT command to retrieve the rows we just inserted from the table.

After we execute the `SELECT` statement, the cursor is something we can loop through in a
for statement. For efficiency, the cursor does not read all of the data from the
database when we execute the `SELECT` statement. Instead, the data is read on
demand as we loop through the rows in the for statement

## **15.5 Structured Query Language summary**

Since there are so many different database vendors, the Structured Query Language
(SQL) was standardized so we could communicate in a portable manner to database
systems from multiple vendors.

A relational database is made up of tables, rows, and columns. The columns
generally have a type such as text, numeric, or date data. When we create a table,
we indicate the names and types of the columns:

```
CREATE TABLE Track (title TEXT, plays INTEGER)
```

To insert a row into a table, we use the SQL INSERT command:

```
INSERT INTO Track (title, plays) VALUES (My Way, 15)
```

The INSERT statement specifies the table name, then a list of the fields/columns
that you would like to set in the new row, and then the keyword VALUES and a list
of corresponding values for each of the fields.

The SQL SELECT command is used to retrieve rows and columns from a database.
The SELECT statement lets you specify which columns you would like to retrieve
as well as a WHERE clause to select which rows you would like to see. It also allows
an optional ORDER BY clause to control the sorting of the returned rows.

```
SELECT * FROM Track WHERE title = My Way
```

Using * indicates that you want the database to return all of the columns for each
row that matches the WHERE clause.

Note, unlike in Python, in a SQL WHERE clause we use a single equal sign to indicate
a test for equality rather than a double equal sign. Other logical operations allowed
in a WHERE clause include <, >, <=, >=, !=, as well as AND and OR and parentheses
to build your logical expressions.

You can request that the returned rows be sorted by one of the fields as follows:

```
SELECT title,plays FROM Track ORDER BY title
```

It is possible to UPDATE a column or columns within one or more rows in a table
using the SQL UPDATE statement as follows:

```
UPDATE Track SET plays = 16 WHERE title = My Way
```

The UPDATE statement specifies a table and then a list of fields and values to change
after the SET keyword and then an optional WHERE clause to select the rows that
are to be updated. A single UPDATE statement will change all of the rows that
match the WHERE clause. If a WHERE clause is not specified, it performs the UPDATE
on all of the rows in the table.

To remove a row, you need a WHERE clause on an SQL DELETE statement. The
WHERE clause determines which rows are to be deleted:
```
DELETE FROM Track WHERE title = My Way
```

## **15.6 Multiple tables and basic data modeling**

The real power of a relational database is when we create multiple tables and make
links between those tables. The act of deciding how to break up your application
data into multiple tables and establishing the relationships between the tables
is called data modeling. The design document that shows the tables and their
relationships is called a data model.

The `JOIN` clause includes an `ON` condition that defines how the rows are to to be
connected. For each row in Track add the data from Artist from the row where
artist_id Track table matches the id from the Artist table.

## **15.7 Data model diagrams**

While there are many graphical representations of data models, we will use one of
the “classic” approaches, called “Crow’s Foot Diagrams” as shown in Figure 15.4.
Each table is shown as a box with the name of the table and its columns. Then
where there is a relationship between two tables a line is drawn connecting the
tables with a notation added to the end of each line indicating the nature of the
relationship.

## **15.8 Automatically creating primary keys**

In the above example, we arbitrarily assigned Frank the primary key of 42. However
when we are inserting millions or rows, it is nice to have the database automatically
generate the values for the id column. We do this by declaring the id column as
a `PRIMARY KEY` and leave out the `id` value when inserting the row:

You can call `SELECT last_insert_rowid()`; after each of the inserts to retrieve
the value that the database assigned to the id of each newly created row.

## **15.9 Logical keys for fast lookup**

If we had a table full of artists and a table full of tracks, each with a foreign key
link to a row in a table full of artists and we wanted to list all the tracks that were
sung by ‘Frank Sinatra’ as follows:

Since we have two tables and a foreign key between the two tables, our data is
well-modeled, but if we are going to have millions of records in the Artist table
and going to do a lot of lookups by artist name, we would benefit if we gave the
database a hint about our intended use of the name column.

We do this by adding an “index” to a text column that we intend to use in WHERE
clauses:

## **15.10 Adding constraints to the database**

We can also use an index to enforce a constraint (i.e. rules) on our database op
erations. The most common constraint is a uniqueness constraint which insists
that all of the values in a column are unique. We can add the optional `UNIQUE`
keyword, to the `CREATE INDEX` statement to tell the database that we would like it
to enforce the constraint on our SQL. We can drop and re-create the artist_name
index with a `UNIQUE` constraint as follows.